# Loading libraries

In [ ]:
import pandas as pd
import numpy as np

# Loading datasets

In [ ]:
## After filtering the 1000 Genomes populational dataset with bcftools to include only missense variants

pop_data_missense = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/populational_data_missense_only.tsv",
                       sep="\t"
                       )


In [ ]:
pop_data_missense.columns.to_list()

# Data processing


In [ ]:
pop_data_missense.rename(columns={
    "'Functional annotations: 'Allele | Annotation | Annotation_Impact | Gene_Name | Gene_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO'":
    "vep_annotation"
    },
                         inplace=True
                         )

In [ ]:
## Processing the hexadecimal representations
pop_data_missense.astype(str).stack().str.contains("x3b").any()


pop_data_missense = pop_data_missense.apply(
    lambda col: col.str.replace(r"\\x3b", "|", regex=True) if col.dtype == "object" else col
)

#### **Process the 8th column**




The dataset's 8th column comprises aggregated functional annotations. The primary objective of this processing phase is to extract and transform this aggregated information into multiple distinct features.

In [ ]:
pop_data_missense.rename(columns={
    "'Functional annotations: 'Allele | Annotation | Annotation_Impact | Gene_Name | Gene_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO'":
    "vep_annotation"
    },
                         inplace=True
                         )

In [ ]:
pop_data_missense_tmp = pop_data_missense.copy()
pop_data_missense_tmp["vep_annotation"] = pop_data_missense_tmp["vep_annotation"].str.split(",")

In [ ]:
def align_column(col, n):
    if pd.isna(col):
        return [pd.NA] * n

    if isinstance(col, str):
        if "/" in col:
            parts = col.split("/")
            return parts[:n] + [pd.NA] * max(0, n - len(parts))
        else:
            return [col] * n

    return [col] * n



rows = []

for _, row in pop_data_missense_tmp.iterrows():
    n = len(row["vep_annotation"])

    new_row = {}
    for col in pop_data_missense_tmp.columns:
        if col == "vep_annotation":
            new_row[col] = row[col]
        else:
            new_row[col] = align_column(row[col], n)

    rows.append(pd.DataFrame(new_row))


pop_data_missense_long = (
    pd.concat(rows)
      .explode("vep_annotation")
      .reset_index(drop=True)
)

In [ ]:
new_headers = [
    "Allele", "Annotation", "Annotation_Impact", "Gene_Name",
    "Gene_ID", "Feature_Type", "Feature_ID", "Transcript_BioType",
    "Rank", "HGVS_c", "HGVS_p", "cDNA_pos_len",
    "CDS_pos_len", "AA_pos_len", "Distance", "ERRORS_WARNINGS_INFO"
]

target_col = "vep_annotation"

split_df = pop_data_missense_long[target_col].str.split('|', expand=True)

split_df = split_df.iloc[:, :len(new_headers)]
split_df.columns = new_headers

col_index = pop_data_missense_long.columns.get_loc(target_col)

part_a = pop_data_missense_long.iloc[:, :col_index]
part_c = pop_data_missense_long.iloc[:, col_index+1:]

pop_data_missense_long_split = pd.concat([part_a, split_df, part_c], axis=1)

pop_data_missense_long_split.head()

In [ ]:
pop_data_missense_long_split[["AA_pos", "AA_len"]] = pop_data_missense_long_split["AA_pos_len"].str.split("/", expand=True)
pop_data_missense_long_split = pop_data_missense_long_split.drop(columns="AA_len")

In [ ]:
# Edit column "HGVS_p"
target_col = 'HGVS_p'

pattern = r'p\.([A-Z][a-z]+)\d+([A-Z][a-z]+)'
new_cols = pop_data_missense_long_split[target_col].str.extract(pattern)
new_cols.columns = ['Ref_AA', 'Alt_AA']  # Name the new columns

col_index = pop_data_missense_long_split.columns.get_loc(target_col)

part_a = pop_data_missense_long_split.iloc[:, :col_index]
part_c = pop_data_missense_long_split.iloc[:, col_index+1:]

pop_data_missense_long_split = pd.concat([part_a, new_cols, part_c], axis=1)

#### **Feature engineering**

In [ ]:
pop_data_missense_long_split = pop_data_missense_long_split.replace('.', np.nan)

In [ ]:
pop_data_missense_long_split.insert(
    0,
    "User_input",
    pop_data_missense_long_split[["CHROM","POS","REF","ALT","Gene_Name"]].astype(str).agg(":".join, axis=1)
)


pop_data_missense_long_split["User_input"] = pop_data_missense_long_split["User_input"].str.replace("chr", "", regex=False)

In [ ]:
pop_data_missense_long_split = pop_data_missense_long_split[pop_data_missense_long_split['Ref_AA'].notna()]

In [ ]:
duplicate = pop_data_missense_long_split['User_input'].value_counts()
duplicate = duplicate[duplicate > 1]
duplicate = duplicate.index.tolist()

In [ ]:
cols_for_values = ["LoFtool_score", "RVIS_percentile", "RVIS", "GDI", "GDI_Phred"]

is_duplicate = pop_data_missense_long_split['User_input'].isin(duplicate)
df_dupes = pop_data_missense_long_split[is_duplicate].copy()
df_unique = pop_data_missense_long_split[~is_duplicate].copy() # Keep the rest safe

df_dupes['priority_score'] = (
    df_dupes[cols_for_values]
    .replace(r'^\s*$', np.nan, regex=True)
    .notna()
    .sum(axis=1)
)

df_dupes = df_dupes.sort_values(by=['User_input', 'priority_score'], ascending=[True, False])

df_resolved = df_dupes.drop_duplicates(subset=['User_input'], keep='first')

df_resolved = df_resolved.drop(columns=['priority_score'])
pop_data_missense_long_split = pd.concat([df_unique, df_resolved], ignore_index=True)

In [ ]:
pop_data_missense_long_split.columns

#### **Subset by the Annovar annotation**

In [ ]:
annovar_columns = pop_data_missense_long_split.iloc[:, 34:134].columns.to_list()
pop_data_missense_annovar = pop_data_missense_long_split[['User_input'] + annovar_columns]
pop_data_missense_long_split = pop_data_missense_long_split.drop(columns=annovar_columns)

#### **Save datasets**

In [ ]:
pop_data_missense_long_split.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/pop_data_missense_final.csv",
                                    sep="\t",
                                    index=False,
                                    na_rep="NA")

pop_data_missense_annovar.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/pop_data_missense_annovar.csv",
                                    sep="\t",
                                    index=False,
                                    na_rep="NA")